# 002 TPU 微调骨架（环境核对 + 单步前向占位，不训练）

本 notebook 是 Kaggle TPU 训练入口的最小骨架，只做三件事：
1. 打印环境 / 设备 / 框架版本；2. 检查数据挂载；3. 单步前向占位。
不启动任何长时间训练，不下载权重与完整训练数据。

**人工操作要求（必读）：**
- 经 CLI 人工 push 启动：`kaggle kernels push -p notebooks/tpu-training --accelerator TpuV5E8`（本轮不 push）。
- 配额：Kaggle TPU 每周 20h、单次上限 9h（2026-09-30 补查本账户剩余 20h 未用，刷新 10-03 00:00 UTC）；实际设备分配以 runtime 为准，配额查询成功不等于分配成功。
- 断点续训：长训练必须将会话上限前把 checkpoint / 优化器 / 配置 / split manifest / seed 保存为可跨会话读取的产物（Kaggle notebook 输出），下次会话恢复。本骨架不实现训练循环，只预留保存路径检查。
- 若 TPU 不可用，再评估 Colab v5e-1 备用路线；不要把本骨架直接当作比赛最终提交入口（最终提交为 CPU/GPU 离线 notebook）。

In [ ]:
import sys, platform
print("python =", sys.version)
print("platform =", platform.platform())

def _ver(mod):
    try:
        m = __import__(mod)
        return getattr(m, "__version__", "installed(version unknown)")
    except Exception as e:
        return f"NOT INSTALLED ({type(e).__name__}: {e})"

for mod in ["torch", "torch_xla", "jax", "pyarrow", "rdkit"]:
    print(f"{mod} =", _ver(mod))

In [ ]:
# 设备探测：torch_xla 缺失时优雅跳过，不报错中断（本地 CPU 照常可跑）
try:
    import torch_xla.core.xla_model as xm
    print("xla devices =", xm.get_xla_supported_devices())
except Exception as e:
    print("TPU probe skipped:", type(e).__name__, str(e)[:200])

try:
    import torch
    print("torch cuda available =", torch.cuda.is_available())
except Exception as e:
    print("torch probe skipped:", type(e).__name__, str(e)[:200])

In [ ]:
# 数据挂载检查：只列目录，不下载 train.parquet（3.03GB）与大权重
from pathlib import Path
for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path("data/raw")]:
    if p.exists():
        kids = sorted([x.name for x in p.iterdir()][:20])
        print(p, "->", kids)
    else:
        print(p, "-> MISSING")

# TPU 训练要求的数据/权重应以私有 Dataset/Model 输入挂载后再在此处断言；
# 本骨架不做断言，只打印可见条目，缺挂载时如实记录即可。

In [ ]:
# 单步前向占位：纯标准库一次小矩阵乘法，证明“单步计算”管线位置；
# 真实 FPNet 单步（d=512/layers=6/heads=8/MAX_PEAKS=128，BCE+排序损失）待 B1/T1 接入。
import random
random.seed(0)
d = 8  # 占位维度（非模型配置）
A = [[random.random() for _ in range(d)] for _ in range(d)]
x = [random.random() for _ in range(d)]
y = [sum(A[i][j] * x[j] for j in range(d)) for i in range(d)]
print("single placeholder forward step done, y[0] =", round(y[0], 6))
print("STOP: no training loop here; wire real model + checkpoint-resume before any budgeted run")